In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "etl_supermercado")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def franja_horaria(hora):
    if hora is None:      
        return "Desconocida"
    elif 6  <= hora < 12: 
        return "Mañana"
    elif 12 <= hora < 18: 
        return "Tarde"
    elif 18 <= hora < 24: 
        return "Noche"
    else:                 
        return "Madrugada"
    
franja_udf = F.udf(franja_horaria, StringType())

In [0]:
def nombre_dia(nom):
    dias = {0:"Domingo",1:"Lunes",2:"Martes",3:"Miércoles",4:"Jueves",5:"Viernes",6:"Sábado"}
    return dias.get(nom, "Desconocido")

dia_udf = F.udf(nombre_dia, StringType())

In [0]:
df_orders  = spark.table(f"{catalogo}.{esquema_source}.orders")
df_op      = spark.table(f"{catalogo}.{esquema_source}.order_products")
df_prod    = spark.table(f"{catalogo}.{esquema_source}.products")
df_aisles  = spark.table(f"{catalogo}.{esquema_source}.aisles")
df_depts   = spark.table(f"{catalogo}.{esquema_source}.departments")

In [0]:
df_orders = df_orders.dropna(how="all").filter(col("order_id").isNotNull())
df_op = df_op.dropna(how="all").filter(col("order_id").isNotNull())

In [0]:
df_joined = df_op.alias("op") \
    .join(df_orders.alias("o"),col("op.order_id") == col("o.order_id"),"inner") \
    .join(df_prod.alias("p"),col("op.product_id") == col("p.product_id"),"left") \
    .join(df_aisles.alias("a"),col("p.aisle_id") == col("a.aisle_id"),"left") \
    .join(df_depts.alias("d"),col("p.department_id") == col("d.department_id"),"left")

In [0]:
df_updated = df_joined \
    .withColumn("franja_horaria",franja_udf(col("o.order_hour_of_day"))) \
    .withColumn("dia_semana",dia_udf(col("o.order_dow"))) \
    .withColumn("es_recompra",col("op.reordered") == 1) \
    .select(
        col("o.order_id"),col("o.user_id"),col("o.eval_set"),col("o.order_number"),
        col("o.order_dow"),col("o.order_hour_of_day"),col("o.days_since_prior_order"),
        col("op.product_id"),col("p.product_name"),col("op.add_to_cart_order"), 
        col("op.reordered"),col("a.aisle_id"),col("a.aisle"),col("d.department_id"),
        col("d.department"),col("franja_horaria"),col("dia_semana"),col("es_recompra"),
        current_timestamp().alias("ingestion_date"),
    )


In [0]:
df_updated.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.ventas_detalle")